# Lesson 02 - Exploring Microsoft Agent Framework

Di **Microsoft Agent Framework (MAF)** na one framework wey dem use build AI agents. E get clean, composable architecture get four core building blocks:

- **Client** – e dey connect to AI model endpoint and e dey handle communication
- **Agent** – e wrap client with instructions and tool definitions
- **Tools** – e extend agent capabilities with custom functions wey di model fit call
- **Session** – e keep conversation history for multi-turn interactions

For dis lesson, we go build **travel booking agent** wey dey check destination availability using dis concepts.


## Setup


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Understanding di Agent Framework Architecture

Di Microsoft Agent Framework dey follow one kind layered architecture:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – One `FoundryChatClient` wey dey connect to Azure OpenAI deployment. E dey handle authentication, request formatting, and response parsing.
2. **Agent** – E dey create from di client through `provider.create_agent()`, di agent dey join model access with instructions (system prompt) and tools.
3. **Tools** – Python functions wey dem put `@tool` for, so dat agent fit use dem do actions or find data.
4. **Session** – One `AgentSession` object (wey dem create via `agent.create_session()`) wey dey store conversation history, e allow multi-turn dialogue where agent fit remember wetin don talk before.

Make we build each layer step by step.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Adding Tools wit di @tool Decorator

Tools dey allow agents to do tins pass just to generate text. Di `@tool` decorator dey change normal Python function to somtin we agent fit call.

Main tins:
- Use `Annotated[type, "description"]` make di model sabi wetin each parameter mean.
- Di docstring go be di tool description we di model go see.
- `approval_mode="never_require"` mean say di tool go run automatically without make user confirm.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Di tin wey you fit take create Agent wit Tools

Now we don join di client, instructions, and tools join body as one agent. Di `instructions` na like system prompt — dem dey yarn how di agent suppose be and how e suppose dey behave.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Multi-Turn Conversations with Sessions

An `AgentSession` (wey dem create wit `agent.create_session()`) dey keep track of all di messages wey dey inside one conversation. If you dey pass di same session to every `agent.run()` call, di agent go get access to di full conversation history and e fit still refer back to di messages wey dem talk before.

We dey pass `tools=[check_destination_availability]` so di agent fit call our availability checker every time e wan do turn.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Summary

For dis lesson, you don explore di four main pions dem wey dey inside Microsoft Agent Framework:

| Concept | Wetin You Learn |
|---------|------------------|
| **Client** | `FoundryChatClient` dey connect to Azure OpenAI wit credential-based auth |
| **Agent** | `provider.create_agent()` combine model connection wit instructions and name |
| **Tools** | Di `@tool` decorator dey expose Python functions make agent fit call am |
| **Session** | `agent.create_session()` dey keep conversation history for plenti turns |

Dis building blocks dem join together make agents wey fit hold natural conversations, call outside functions, and keep context — na di foundation for more advanced agentic patterns for di lessons wey dey come later.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
